In [ ]:
import uproot
import awkward as ak

import pandas as pd

import matplotlib.pylab as plt
import numpy as np

import hist
from hist import Hist

import time

import myPIDselector

import math

import os

data = ak.from_parquet("Background_and_signal_SP_modes_All_runs.parquet")

plt.close()





###################################################################################
def calculate_bits_for_PID_selector(trkidx, trk_selector_map, verbose=0):
    
    bits = None

    # If there is no trk index passed in, just calculate the bits for
    # all of the tracks
    if trkidx is None:
        trkidx = ak.local_index(trk_selector_map)

    # Grab the tracks that map on to the particle/collection we are interested in 
    subset_of_trk_selector_map = trk_selector_map[trkidx]
    if verbose:
        print("values in the subset of the trk selector map")
        print(subset_of_trk_selector_map)
        print()
        
    # We need this to convert the numbers in the selector to binary
    binary_repr_vec = np.vectorize(np.binary_repr)

    # Grab the number of entries in each so we can unflatten this later
    counts = ak.num(subset_of_trk_selector_map)
    
    # Now get the binary representation (as a string) for the flattened subset
    binrep = binary_repr_vec(ak.flatten(subset_of_trk_selector_map), width=32)

    if verbose:
        print("binary representation of selector map")
        print(binrep)
        print()

    # Convert the string to integers
    tempbits = np.array(binrep).astype(int)
    bits = ak.unflatten(tempbits,counts)

    if verbose:
        print("flattened integer representation of selectors as binary (int)")
        print(tempbits)
        print()
        print("unflattened integer representation of selectors as binary (int)")
        print(bits)
        print()

    return bits

##########################################################################

##########################################################################

def mask_PID_selection(bits, selector, pid_map_object):

    bit_to_look_for = pid_map_object.selectors.index(selector)

    place = int(math.pow(10,bit_to_look_for))
    #print(place)

    mask = bits // place % 10

    mask_bool = ak.values_astype(mask,bool)

    return mask_bool



def get_info_for_PID_masks(ak_arr, verbosity=0):

    # Proton and pion information from the Lambda decay
    # These are the index of the proton (d1) and pion (d2) in those lists
    L1d1idx = ak_arr.Lambda0d1Idx[ak_arr.Bd1Idx]
    L1d2idx = ak_arr.Lambda0d2Idx[ak_arr.Bd1Idx]

    L2d1idx = ak_arr.Lambda0d1Idx[ak_arr.Bd2Idx]
    L2d2idx = ak_arr.Lambda0d2Idx[ak_arr.Bd2Idx]


    
     
    #d1lund = ak_arr['Lambda0d1Lund']
    #d2lund = ak_arr['Lambda0d2Lund']
    
    #Bd2idx = ak_arr['Bd2Idx']
    #Bd2lund = ak_arr['Bd2Lund']
    
    trkidx_proton = data['pTrkIdx']
    trk_selector_map_proton = data['pSelectorsMap']
    
    trkidx_pion = data['piTrkIdx']
    trk_selector_map_pion = data['piSelectorsMap']

    indices_and_maps = {}
    indices_and_maps['L1d1idx'] = L1d1idx
    indices_and_maps['L1d2idx'] = L1d2idx
    indices_and_maps['L2d1idx'] = L2d1idx
    indices_and_maps['L2d2idx'] = L2d2idx
    #indices_and_maps['Bd2idx'] = Bd2idx

    indices_and_maps['trkidx_proton'] = trkidx_proton
    indices_and_maps['trk_selector_map_proton'] = trk_selector_map_proton

    indices_and_maps['trkidx_pion'] = trkidx_pion
    indices_and_maps['trk_selector_map_pion'] = trk_selector_map_pion

    return indices_and_maps


def PID_masks(indices_and_maps, \
              lam1p_selector,\
              lam1pi_selector, \
              lam2p_selector,\
              lam2pi_selector, \
             verbosity=0):
                #change Bpselector
    L1d1idx = indices_and_maps['L1d1idx']
    L1d2idx = indices_and_maps['L1d2idx']
    L2d1idx = indices_and_maps['L2d1idx']
    L2d2idx = indices_and_maps['L2d2idx']
    #Bd2idx = indices_and_maps['Bd2idx']

    trkidx_proton = indices_and_maps['trkidx_proton']
    trk_selector_map_proton = indices_and_maps['trk_selector_map_proton']

    trkidx_pion = indices_and_maps['trkidx_pion']
    trk_selector_map_pion = indices_and_maps['trk_selector_map_pion']
    
    # Proton
    pbits = calculate_bits_for_PID_selector(trkidx_proton, trk_selector_map_proton, verbose=verbosity)
    # Pion
    pibits = calculate_bits_for_PID_selector(trkidx_pion, trk_selector_map_pion, verbose=verbosity)
    
    
    #selector_proton = 'TightKMProtonSelection'
    #selector_pion = 'TightKMPionMicroSelection'
    #print(f"Now trying to create a mask with {selector_proton}")
    #print(f"Now trying to create a mask with {selector_pion}")
    
    
    mask_bool_protonL1 = mask_PID_selection(pbits[L1d1idx], lam1p_selector, pps)
        
    mask_bool_pionL1 = mask_PID_selection(pibits[L1d2idx], lam1pi_selector, pips)

    mask_bool_protonL2 = mask_PID_selection(pbits[L2d1idx], lam2p_selector, pps)
        
    mask_bool_pionL2 = mask_PID_selection(pibits[L2d2idx], lam2pi_selector, pips)

    return mask_bool_protonL1, mask_bool_pionL1,mask_bool_protonL2, mask_bool_pionL2



pps = myPIDselector.PIDselector("p")
pips = myPIDselector.PIDselector("pi")


for myPMask in ["SuperLooseKMProtonSelection"]:
    for myPiMask in ["SuperLooseKMPionMicroSelection"]: 

        indices_and_maps = get_info_for_PID_masks(data, verbosity=0)

        mask_bool_protonL1, mask_bool_pionL1,mask_bool_protonL2, mask_bool_pionL2 = PID_masks(indices_and_maps, \
                            lam1p_selector=myPMask, \
                            lam1pi_selector=myPiMask, \
                            lam2p_selector=myPMask, \
                            lam2pi_selector=myPiMask, \
                            verbosity=0)

        for valu in ['998','-999']:
            fiveCuts = []
            tenCuts = []
            twentyCuts = []
            fivePeaks = []
            fiveBkg=[]
            tenBkg = []
            twentyBkg=[]
            tenPeaks = []
            twentyPeaks = []
            fiveSig = []
            tenSig = []
            twentySig = []
            fiveAllentries = []
            fivePcts = []
            fiveCuts = []
            fiveTags = []
            tenAllentries = []
            tenPcts = []
            tenCuts = []
            tenTags = []
            twentyAllentries = []
            twentyPcts = []
            twentyCuts = []
            twentyTags = []
            for minCut in [5,10,20]:
                valid = []
                inValid = []
                passCount = 0
                valCount = 0
                inValCount=0
                emptyCount=0
                spmask = (data['spmode']==valu)

                #mes = data['BpostFitMes']
                #de = data['BpostFitDeltaE']

                #meslo = INT32_MIN
                #meshi =INT32_MAX
                #DeltaElo = INT32_MIN
                #DeltaEhi = INT32_MAX
                mask_pid = mask_bool_pionL1 & mask_bool_protonL1 & mask_bool_pionL2 & mask_bool_protonL2 
                mask_pid_event = ak.any(mask_pid, axis=1)
                mask_fit = mask_pid_event #(mes>meslo) & (mes<meshi) & (de>DeltaElo) & (de<DeltaEhi)
                #mask = mask & mask_fit
                mask_fit = mask_fit[spmask]


                lamconflsig = data[spmask]['Lambda0postFitFlightSignificance'][mask_fit]
                lamfl_basic = data[spmask]['Lambda0FlightLen'][mask_fit]
                lamuncmass = data[spmask]['Lambda0_unc_Mass'][mask_fit]
                numBary = data[spmask]['nB'][mask_fit]
                

                flight_tag = 'lampostfitsig'

                lammass_world_average = 1.115683
                width = 0.003
                lo = lammass_world_average - width
                hi = lammass_world_average + width

                
                

                

                nrows = 8
                ncols = 8


                #plt.hist(ak.flatten(m, axis=None),      bins=100,range=(1.105, 1.125))
                #plt.figure()
                #plt.hist(mSect1First, bins=100, range=(1.105, 1.125))
                #plt.hist(mSect1Second, bins=100, range=(1.105, 1.125))

                
                fig1,axes1 = plt.subplots(figsize=(12,8),nrows=nrows, ncols=ncols, sharex=True, sharey=True)
                #fig2,axes2 = plt.subplots(figsize=(12,8),nrows=nrows, ncols=ncols, sharex=True, sharey=True,)
                
                for i in range(minCut,60):

                    m = lamuncmass

                    mask_fl = (lamconflsig>minCut)&(numBary==1)
                    #diff cut upper and lower
                    mask_upper_fl = (lamconflsig>i)
                    m = m[mask_fl&(ak.any(mask_upper_fl, axis=1))]

                    #m = m[mask_fl]

                    print(len(m))

                    sizeMask = (ak.num(m, axis=1) == 2)

                    m = m[sizeMask]
                    print(len(m))



                    #mask = (m>lo)&(m<hi)&mask_fl
                    #mask_lo_sideband = (m<=lo) & (m>(lo-2*width)) & mask_fl
                    #mask_hi_sideband = (m>=hi) & (m<(hi+2*width)) & mask_fl

                    outerMask = (((m[:,0]<lo) | (m[:,0]>hi)) & ((m[:,1]<lo) | (m[:,1]>hi)))
                    
                    b0 = len(m[outerMask])/4

                    s1Mask = ((m[:,0]<lo)&(lo<m[:,1])&(m[:,1]<hi))
                    s2Mask = ((m[:,0]>hi)&(lo<m[:,1])&(m[:,1]<hi))
                    s3Mask = ((m[:,1]<lo)&(lo<m[:,0])&(m[:,0]<hi))
                    s4Mask = ((m[:,1]>hi)&(lo<m[:,0])&(m[:,0]<hi))


                    b1 = (len(m[s1Mask])+len(m[s2Mask])+len(m[s3Mask])+len(m[s4Mask]))/4 - b0

                    s5Mask = (((lo<m[:,0])&(m[:,0]<hi))&((lo<m[:,1])&(m[:,1]<hi)))

                    mass_cand_out_0 = m[outerMask][:, 0]
                    mass_cand_out_1 = m[outerMask][:, 1]

                    mass_cand_s1_0 = m[s1Mask][:, 0]
                    mass_cand_s1_1 = m[s1Mask][:, 1]

                    mass_cand_s2_0 = m[s2Mask][:, 0]
                    mass_cand_s2_1 = m[s2Mask][:, 1]

                    mass_cand_s3_0 = m[s3Mask][:, 0]
                    mass_cand_s3_1 = m[s3Mask][:, 1]

                    mass_cand_s4_0 = m[s4Mask][:, 0]
                    mass_cand_s4_1 = m[s4Mask][:, 1]

                    mass_cand_s5_0 = m[s5Mask][:, 0]
                    mass_cand_s5_1 = m[s5Mask][:, 1]

                    all_x = [mass_cand_out_0, mass_cand_s1_0, mass_cand_s2_0,mass_cand_s3_0, mass_cand_s4_0, mass_cand_s5_0]
                    all_y = [mass_cand_out_1, mass_cand_s1_1, mass_cand_s2_1,mass_cand_s3_1, mass_cand_s4_1, mass_cand_s5_1]

                    # 2. Concatenate them into single flat NumPy arrays
                    x_combined = ak.to_numpy(ak.concatenate(all_x))
                    y_combined = ak.to_numpy(ak.concatenate(all_y))

                    # 2. Create a 2D Histogram (highly recommended for large particle physics datasets)
                    if False:
                        plt.figure(figsize=(8, 6))
                        plt.hist2d(
                            x_combined, 
                            y_combined, 
                            bins=100, 
                            range=[[lo-1.05*width, hi+1.05*width], [lo-1.05*width, hi+1.05*width]], 
                            cmap='viridis'
                        )

                        # 3. Add labels and a colorbar
                        plt.colorbar(label='Candidates per bin')
                        plt.xlabel(r'$\Lambda$ Candidate 1 Mass [GeV/$c^2$]')
                        plt.ylabel(r'$\Lambda$ Candidate 2 Mass [GeV/$c^2$]')
                        plt.title('2D Mass Correlation')

                        plt.show()
                    

                    nsig = len(m[s5Mask]) - b1 - b0

                    nbkg = b1+b0
                    npeak = len(m[s5Mask])
                    nall = len(lamuncmass[lamfl_basic>=0]) #?????? Should this be events with 2 lambda, events with enough fltsig, or all events?

                    #nall = len(ak.flatten(m[lamfl_basic>=0], axis=None))


                    #npeak = len(ak.flatten(m[mask], axis=None))
                    #nbkglo = len(ak.flatten(m[mask_lo_sideband], axis=None))
                    #nbkghi = len(ak.flatten(m[mask_hi_sideband], axis=None))
                    #nbkg = (nbkglo + nbkghi)/2.0
                    #nsig = npeak - nbkg

                    #print(f"nall: {nall}     npeak: {npeak}   nbkg: {nbkg}     nsig: {nsig}   {nbkglo}   {nbkghi}")
                    if minCut == 5:
                        fivePeaks.append(npeak)
                        fiveSig.append(nsig)
                        fiveBkg.append(nbkg)
                        fiveCuts.append(i)
                        fiveAllentries.append(nall)
                        fiveTags.append(flight_tag)
                    elif minCut == 10:
                        tenPeaks.append(npeak)
                        tenSig.append(nsig)
                        tenBkg.append(nbkg)
                        tenCuts.append(i)
                        tenAllentries.append(nall)
                        tenTags.append(flight_tag)
                    elif minCut == 20:
                        twentyPeaks.append(npeak)
                        twentySig.append(nsig)
                        twentyBkg.append(nbkg)
                        twentyCuts.append(i)
                        twentyAllentries.append(nall)
                        twentyTags.append(flight_tag)

                    row = int(i/ncols)
                    col = i%ncols
                    
                    plt.sca(axes1[row][col])
                        
                    plt.hist(ak.flatten(lamuncmass, axis=None),bins=100,range=(1.105, 1.125), label=f'flcut>{i:.1f}')
                    plt.hist(ak.flatten(m, axis=None),      bins=100,range=(1.105, 1.125))
                    #plt.hist(ak.flatten(m[mask_lo_sideband], axis=None),bins=100,range=(1.105, 1.125), color='yellow')
                    #plt.hist(ak.flatten(m[mask_hi_sideband], axis=None),bins=100,range=(1.105, 1.125), color='yellow')
                    plt.legend()

                    #plt.sca(axes2[row][col])
                    if row == nrows-1:
                        axes1[row][col].set_xlabel(r'$\Lambda^0$ mass [GeV/c$^2$]')
                        #axes2[row][col].set_xlabel(r'$\Lambda^0$ flight len [cm]')

                    fllo, flhi = 0,30
                    if flight_tag=='lampostfitsig':
                        fllo, flhi = 0,100

                    #plt.hist(ak.flatten(lamconflsig, axis=None),         bins=100,range=(fllo, flhi))
                    #plt.hist(ak.flatten(lamconflsig[mask_fl], axis=None),bins=100,range=(fllo, flhi))

                #fig1.subplots_adjust(wspace=0, hspace=0)#left=0.1, right=0.1, bottom=0.1, top=0.1, wspace=0.1, hspace=0.1)#wspace=0, hspace=0)
                #fig2.subplots_adjust(wspace=0, hspace=0)#left=0.1, right=0.1, bottom=0.1, top=0.1, wspace=0.1, hspace=0.1)#wspace=0, hspace=0)
                
                #fig1.subplots_adjust(0,0,1,1,0,0)
                #fig2.subplots_adjust(0,0,1,1,0,0)

                #plt.figure(fig1)
                #plt.suptitle(f'{flight_tag}')
                #plt.figure(fig2)
                #plt.suptitle(f'{flight_tag}')
                
                #fig1.tight_layout()
                #fig2.tight_layout()
                

            fivesigs = np.array(fiveSig)
            fivebkgs = np.array(fiveBkg)
            fivepeaks = np.array(fivePeaks)
            fivecuts = np.array(fiveCuts)
            fiveallentries = np.array(fiveAllentries)

            tensigs = np.array(tenSig)
            tenbkgs = np.array(tenBkg)
            tenpeaks = np.array(tenPeaks)
            tencuts = np.array(tenCuts)
            tenallentries = np.array(tenAllentries)

            twentysigs = np.array(twentySig)
            twentybkgs = np.array(twentyBkg)
            twentypeaks = np.array(twentyPeaks)
            twentycuts = np.array(twentyCuts)
            twentyallentries = np.array(twentyAllentries)

            fivepcts = fivesigs/fivesigs[0] * 100
            fivebkg_under_peak = fivepeaks - fivesigs
            fivepct_bkg_under_peak = fivebkg_under_peak / fivebkg_under_peak[0] * 100

            tenpcts = tensigs/tensigs[0] * 100
            tenbkg_under_peak = tenpeaks - tensigs
            tenpct_bkg_under_peak = tenbkg_under_peak / tenbkg_under_peak[0] * 100

            twentypcts = twentysigs/twentysigs[0] * 100
            twentybkg_under_peak = twentypeaks - twentysigs
            twentypct_bkg_under_peak = twentybkg_under_peak / twentybkg_under_peak[0] * 100

            S5 = fivesigs
            B5 = fivebkg_under_peak
            scale = 1
            fom5 = scale*S5/np.sqrt(scale*S5 + scale*B5)

            S10 = tensigs
            B10 = tenbkg_under_peak
            scale = 1
            fom10 = scale*S10/np.sqrt(scale*S10 + scale*B10)

            S20 = twentysigs
            B20 = twentybkg_under_peak
            scale = 1
            fom20 = scale*S20/np.sqrt(scale*S20 + scale*B20)

            #max_fom = max(fom[7:])
            #idx_max = fom.tolist().index(max_fom)
            #print(valu, "/", myPMask, myPiMask)
            #print(f"idx_max: {idx_max}  max cuts: {cuts[idx_max]:.2f} max fom: {fom[idx_max]:.2f}   pcts: {pcts[idx_max]:.2f}")


            plt.figure(figsize=(12, 8))
            plt.subplot(3,2,1)
            plt.plot(fivecuts, fivepeaks,'o',color='blue')
            plt.plot(tencuts, tenpeaks,'o',color='orange')
            plt.plot(twentycuts, twentypeaks,'o',color='green')
            plt.ylabel('# under peak')
            plt.xlabel(f'Cut on flight sig value')

            plt.subplot(3,2,2)
            plt.plot(fivecuts, fivesigs,'o',color='blue')
            plt.plot(tencuts, tensigs,'o',color='orange')
            plt.plot(twentycuts, twentysigs,'o',color='green')
            #plt.ylabel('# of signal in peak (# in peak - # est background')
            plt.ylabel('# of signal in peak')
            plt.xlabel(f'Cut on flight sig value')

            plt.subplot(3,2,3)
            plt.plot(fivecuts, fivebkgs,'o',color='blue')
            plt.plot(tencuts, tenbkgs,'o',color='orange')
            plt.plot(twentycuts, twentybkgs,'o',color='green')
            plt.ylabel('# est background')
            plt.xlabel(f'Cut on flight sig value')
                
            plt.subplot(3,2,4)
            plt.plot(fivecuts, fivepcts,'o',color='blue')
            plt.plot(tencuts, tenpcts,'o',color='orange')
            plt.plot(twentycuts, twentypcts,'o',color='green')
            #plt.ylim(0.1,1.1)
            plt.ylabel('% sig remaining')
            plt.xlabel(f'Cut on flight sig value')
                
            plt.subplot(3,2,5)
            plt.plot(fivecuts, fivepct_bkg_under_peak,'o',color='blue')
            plt.plot(tencuts, tenpct_bkg_under_peak,'o',color='orange')
            plt.plot(twentycuts, twentypct_bkg_under_peak,'o',color='green')
            #plt.ylim(0.7,1.1)
            plt.ylabel('% bkg under peak')
            plt.xlabel(f'Cut on flight sig value')
                
            # Naive significance
            # Multiply by 10 if we are only doing Run 1
            plt.subplot(3,2,6)
            #plt.plot(cuts, 10*sigs/np.sqrt(10*bkg_under_peak),'o')
            plt.plot(fivecuts, fom5,'o',color='blue')
            plt.plot(tencuts, fom10,'o',color='orange')
            plt.plot(twentycuts, fom20,'o',color='green')
            plt.ylabel(r'$S/\sqrt{ S + B }$')
            plt.xlabel(f'Cut on flight sig value')

            plt.suptitle(f'{valu} / loCut = 5,10,20')

            plt.tight_layout()
                
            plt.legend() 
            plt.show()

"""
plt.figure()
plt.hist(ak.flatten(lamuncmass),bins=100)
plt.xlabel(f'Lambda^0 mass (GeV/c$^2$)', fontsize=18)

plt.figure()    
plt.hist(ak.flatten(lamconflsig),bins=100, range=(0,120))
plt.xlabel(f'Lambda^0 post-fit flight length significance')

plt.show()
"""